# C1.8 · Defensive deception and threshold failures

**Function C — Agentic Evaluation and Red Teaming → One Red-Team Lifecycle, End to End**

Builds on **[C1.7 · Triaging the non-deterministic swarm](https://spbreed.github.io/cyber-commons/lessons/C1.7.html)**.

| | |
|---|---|
| Tools used | Canarytokens |

## What this lesson is

**What it covers.** Placing canary tokens and honeypot tasks in a data index so a harvesting agent trips a zero-false-positive alert, and checking nothing legitimate reaches the bait.

**Why a security engineer needs it.** Every other detector trades misses against false alarms through a threshold. A canary has none — unless it is placed where real work touches it, which converts it back into a tuned control.

## 1 · The hook

Every detector in this function needed a threshold, and every threshold is a trade. A canary in the index needs neither — nothing legitimate has any reason to touch it — until the canary is placed where real work does.

> **At CyberTravels.** The canary is a fake booking record in CyberTravels' index that no legitimate itinerary references, so a read of it is an agent going somewhere its task never sent it.

## 2 · The framework

```
   every other detector          deception

   signal -> threshold -> alert  canary token in the index
             (a trade)                   |
                                   nothing legitimate reads it
                                         |
                                   one touch = alert, FP rate 0
   failure: a canary placed where real work reaches it, which
   converts a zero-FP control back into a tuned one
```

Every detector in this function has needed a threshold, and every threshold is a
trade. **Deception is the exception**: a canary token or a honeypot task in a
data index has a false-positive rate of zero by construction, because nothing
legitimate has any reason to touch it.

Defensive deception means placing weaponised values — tokens that look genuine,
files no real task references — where a data-harvesting agent would plausibly
reach, so the alert needs no threshold at all. The failure mode is a canary that
legitimate work does touch, which converts a zero-false-positive control back
into a tuned one.

## 3 · The procedure, as a skill

The skill places canary tokens and a honeypot task in CyberTravels' environment, checks that no legitimate path reaches them, and reads a touch as a zero-false-positive signal.

### The skill — [`skills/detection/canary-and-honeypot-design/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/canary-and-honeypot-design/SKILL.md)

```yaml
name: canary-and-honeypot-design
description: >-
  Place credential canaries and honeypot tasks in an agent's environment so that
  an alert has no structurally possible false positive, and measure how fast the
  signal decays as agents learn. Use when deception is being added, or when a
  detection needs to be one nobody has to triage.
allowed-tools: Read, Grep, Glob
```

# An alert with no possible false positive

A canary credential that nothing legitimate uses produces an alert that needs no
triage: the only way it authenticates is that somebody read it and tried it.
That property is structural, not statistical, and it is why deception belongs in
an agent environment where every other signal is ambiguous.

## When to use this

Designing detection for agent environments, and whenever an existing detection's
false-positive rate is the reason it is ignored.

## Procedure

**1 — Place canaries where only reading them is unusual.** Environment
variables, config files, the fixtures a code agent walks. They must be
indistinguishable from real ones — a canary named `canary_key` is a filter, not
a trap.

**2 — Ensure nothing legitimate uses them.** This is the whole property. Check
the code, the tests and the deployment. One legitimate reference and the alert
becomes triage.

**3 — Instrument the authentication path** to capture source address and user
agent on use. A canary alert with no context tells you that it happened and
nothing about who.

**4 — Add honeypot tasks for behaviour rather than credentials.** A task with an
available shortcut that nothing legitimate would take. Log the attempt and score
it; this measures inclination, which no credential can.

**5 — Measure decay.** Agents and operators learn. Model the hit rate over days
since placement, with and without rotation, and set the rotation interval from
the curve rather than from a calendar.

## Example

**Input** — the fixture committed at the top of [`scripts/canary_and_honeypot_design.py`](scripts/canary_and_honeypot_design.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
token                     source        agent                 verdict
hf_liveTokenNotShown      10.2.0.11     ci-runner             normal use
hf_CANARY7Fq2mXvLpR8s     203.0.113.9   python-requests/2.31  CONFIRMED COMPROMISE
ghp_alsoLive              10.2.0.11     ci-runner             normal use
sk-CANARYd3Vn8yHc2Uae     203.0.113.9   python-requests/2.31  CONFIRMED COMPROMISE

canary hits: 2  false positives possible: 0
Not zero because the detector is good - zero because nothing legitimate
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "canaries": [{"id": "str", "placed_in": "str", "indistinguishable": true, "legitimate_refs": 0}],
  "alerts": [{"canary": "str", "source_ip": "str", "user_agent": "str", "false_positive_possible": false}],
  "honeypot_tasks": [{"task": "str", "shortcut": "str", "attempts": 0}],
  "decay": {"days": [0], "hit_rate": [0.0], "rotation_days": 0}
}
```

## Failure modes

- **A canary anything legitimate touches.** The property is gone and the alert
  becomes noise.
- **Naming it as a canary.** It becomes a filter for the competent attacker.
- **Never rotating.** The signal decays and the absence of alerts reads as
  safety.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/canary-and-honeypot-design/scripts/canary_and_honeypot_design.py
SCRIPT = "skills/detection/canary-and-honeypot-design/scripts/canary_and_honeypot_design.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

The canaries sit outside every legitimate path, so a single touch is a high-confidence alert with no threshold, and the one placed too close to real work is flagged as a false-positive source before it ships.

## Your turn

Plant one canary credential in a place only an over-reaching agent would look, and wire its use to a page. It is the cheapest high-signal detector you will build.

---

**Next → [C1.9 · Machine-speed containment and fleet revocation](https://spbreed.github.io/cyber-commons/lessons/C1.9.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.8.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.8.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*